In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#load the dataset
data=pd.read_csv('../../artifacts/Churn_Modelling.csv')
data.head()

In [ ]:
data.info()

#### Preprocess the data

In [ ]:
data.drop(columns=['RowNumber',"CustomerId","Surname"],inplace=True, axis=1)

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data.isnull().sum()

In [ ]:
data["Geography"].value_counts()

In [ ]:
data["Gender"].value_counts()

In [ ]:
numerical_features=list(data.select_dtypes(exclude='object').columns)
print(numerical_features)

In [ ]:
# EstimatedSalary is our target feature so we will exclude EstimatedSalary
numerical_features.remove('EstimatedSalary')
print(numerical_features)

In [ ]:
Ohe_feature=['Geography','Gender']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_df,test_df=train_test_split(data,random_state=43,test_size=0.2)

In [ ]:
train_df.shape

In [ ]:
X_train=train_df.drop('EstimatedSalary',axis=1)
X_test=test_df.drop('EstimatedSalary',axis=1)
y_train=train_df['EstimatedSalary']
y_test=test_df['EstimatedSalary']

In [ ]:
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
preprocessor=ColumnTransformer(
    transformers=[
        ('standardizer',StandardScaler(),numerical_features),
        ('one hot encoding',OneHotEncoder(handle_unknown='ignore'),Ohe_feature),
    ],
    remainder='passthrough'
)

In [ ]:
X_train=preprocessor.fit_transform(X_train)

In [ ]:
X_test=preprocessor.transform(X_test)

In [ ]:
column_names=preprocessor.get_feature_names_out()
print(column_names)

#### gender and geography columns are removed. 2 columns from gender are added and 3 columns from geography are added. So 11-2+5=14 columns after transformation

In [ ]:
X_test.shape,y_test.shape

In [ ]:
X_train.shape,y_train.shape

In [ ]:
## save the preproceesor
filename="../../artifacts/regression/preprocessor.pkl"
with open(filename,'wb') as file:
    pickle.dump(preprocessor,file)

### ANN implementation

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [ ]:
# Build Model
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),# HL1
    Dense(32,activation='relu'),# HL2
    Dense(1), # Output Layer
])

In [ ]:
model.summary()

In [ ]:
from tensorflow.keras.optimizers import Adam
opt=Adam(learning_rate=0.01)

In [ ]:
# compile the model
model.compile(optimizer=opt,loss="binary_crossentropy",metrics=["accuracy"])

In [ ]:
# setup the tensor board
log_dir="logs/fit/"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

In [ ]:
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [ ]:
# setup up early stopping
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [ ]:
# train the model
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

In [ ]:
model.save('../../artifacts/regression/model.h5')

In [ ]:
# Load tensorboard Extension
%load_ext tensorboard

In [ ]:
%tensorboard --logdir logs/fit